# Recipe embeddings & search (Colab-ready)

This notebook demonstrates an end-to-end recipe embedding pipeline using `sentence-transformers` and a local FAISS search demo. It saves `recipe_embeddings.npy` and `recipes_with_meta.parquet` which you can copy to Google Drive or GCS for later server use.

In [ ]:
# 1) Environment & Dependencies

# Run this cell in Colab (or locally) to install required packages
!pip install -q sentence-transformers pandas numpy pyarrow faiss-cpu scikit-learn matplotlib seaborn joblib pytest

# Assert versions
import sys
import sentence_transformers as st
import pandas as pd
import numpy as np
print('Python', sys.version)
print('pandas', pd.__version__)
print('numpy', np.__version__)
print('sentence-transformers', st.__version__)


In [ ]:
# 2) Imports & Config
import os
import random
import logging
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('recipe-embed')


In [ ]:
# 3) Load Data
# Place Food_Recipe.csv into the same folder as this notebook (or update the path)
DATA_PATH = 'Food_Recipe.csv'

if not Path(DATA_PATH).exists():
    print(f"Warning: {DATA_PATH} not found. Upload your CSV to the notebook runtime or mount Drive.")

# Read and basic checks
df = pd.read_csv(DATA_PATH)
print('Rows:', len(df))
print('Columns:', df.columns.tolist())

# The code expects at least: 'description' or 'recipe_name' (title), 'ingredients_name', 'ingredients_quantity', 'instructions'
required_cols = ['ingredients_name']
for c in required_cols:
    if c not in df.columns:
        raise ValueError(f"Missing required column: {c}")

# Drop rows missing ingredients
df = df.dropna(subset=['ingredients_name']).reset_index(drop=True)
print('After dropping missing ingredients:', len(df))

# Preview
df.head()


In [ ]:
# 4) Feature engineering: Create title and name_for_embedding
# Use `description` or `recipe_name` for title if available
if 'description' in df.columns:
    df['title'] = df['description']
elif 'recipe_name' in df.columns:
    df['title'] = df['recipe_name']
else:
    df['title'] = 'Untitled'

# Combine ingredient name + quantity for better embeddings
if 'ingredients_quantity' in df.columns:
    df['name_for_embedding'] = df['ingredients_name'].astype(str) + ' ' + df['ingredients_quantity'].astype(str)
else:
    df['name_for_embedding'] = df['ingredients_name'].astype(str)

# Sanity check
print('Sample:', df[['title','name_for_embedding']].head().to_dict(orient='records')[:3])


In [ ]:
# 5) Compute embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')

names = df['name_for_embedding'].astype(str).tolist()

# Batch encode
embeddings = model.encode(names, batch_size=32, show_progress_bar=True)
embeddings = np.array(embeddings, dtype=np.float32)

print('Embeddings shape:', embeddings.shape)

# Save artifacts
np.save('recipe_embeddings.npy', embeddings)
df.to_parquet('recipes_with_meta.parquet', index=False)
print('Saved recipe_embeddings.npy and recipes_with_meta.parquet')


In [ ]:
# 6) Upload to Google Drive (optional)
# Mount Drive and copy artifacts to your Drive for later retrieval
from google.colab import drive

drive.mount('/content/drive')
!cp recipe_embeddings.npy recipes_with_meta.parquet /content/drive/MyDrive/ || true
print('Files copied to Drive (if mounted and path exists)')


In [ ]:
# 7) FAISS demo: load artifacts and run a small search
import faiss

# Load
embs = np.load('recipe_embeddings.npy')
meta = pd.read_parquet('recipes_with_meta.parquet')
print('Loaded embeddings', embs.shape, 'and metadata', meta.shape)

# Normalize for cosine
faiss.normalize_L2(embs)

# Build index
d = embs.shape[1]
index = faiss.IndexFlatIP(d)
index.add(embs)

# Helper function
from sklearn.metrics.pairwise import cosine_similarity as sk_cos

def search(query, top_k=5):
    qv = model.encode([query]).astype('float32')
    faiss.normalize_L2(qv)
    D, I = index.search(qv, top_k)
    results = []
    for score, idx in zip(D[0], I[0]):
        results.append({
            'score': float(score),
            'title': meta.iloc[idx]['title'],
            'ingredients': meta.iloc[idx]['ingredients_name'],
            'instructions': meta.iloc[idx].get('instructions', '')
        })
    return results

# Try it
print(search('cooked rice pineapple green beans', top_k=3))


# 8) Quick notes: usage & next steps

# - Use higher-volume batch_size if you have many rows.
# - For production, either upload the .npy/.parquet to GCS or use a managed vector DB (Pinecone, Milvus) and deploy a small FastAPI server that loads the artifacts and exposes a /search endpoint.
# - Consider normalizing ingredient names and units ("1 cup" -> "1 cup", lemmatization) for better matching.

print('Notebook ready: run the cells in order to compute and save embeddings')
